# 02 — Data Loading

**Purpose:** Load ELSA waves 6, 7, 8 from Stata `.dta` files, extract the agreed feature set,
merge on `idauniq`, and export a clean longitudinal panel for downstream preprocessing.

**Inputs:**
- `wave_X_elsa_data*.dta` — core interview data (CES-D items + variables not in derived files)
- `wave_X_ifs_derived_variables.dta` — IFS derived variables (main feature source)
- `wave_X_financial_derived_variables.dta` — income/wealth quintiles only

**Outputs:**
- `outputs/panel_raw.parquet` — wide-format panel, one row per participant, all waves
- `outputs/loading_report.txt` — row counts, merge stats, missingness summary

**Runtime:** Works on Colab, local macOS/Linux, and HPC.  
Set `ELSA_DATA_ROOT` environment variable on HPC/lab machines, or edit `config.py`.

## 0 — Environment setup

Run the install cell only if packages are missing (Colab / fresh HPC environment).  
Skip on local if you've already activated the conda environment.

In [ ]:
# Install dependencies if needed (safe to re-run; skips if already installed)
import importlib, subprocess, sys

REQUIRED = {
    "pyreadstat": "pyreadstat",
    "pandas":     "pandas",
    "numpy":      "numpy",
}

for module, pkg in REQUIRED.items():
    if importlib.util.find_spec(module) is None:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print("Dependencies OK")

In [ ]:
# ── Colab: mount Google Drive (skipped automatically on other runtimes) ──────
import sys
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules or Path("/content").exists()

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("Drive mounted")
else:
    print("Not on Colab — skipping Drive mount")

In [ ]:
# ── Add repo root to path so config.py is importable ─────────────────────────
import sys
from pathlib import Path

# Works whether the notebook is in notebooks/02_data_loading/ or repo root
REPO_ROOT = Path("__file__").resolve().parents[1] if Path("__file__").exists() else Path.cwd()
# Colab: notebooks are often run from /content, so try one level up too
for candidate in [REPO_ROOT, REPO_ROOT.parent, Path.cwd(), Path.cwd().parent]:
    if (candidate / "config.py").exists():
        REPO_ROOT = candidate
        break

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"REPO_ROOT: {REPO_ROOT}")

In [ ]:
import config
config.check_paths()   # prints ✓/✗ for every expected data file

## 1 — Imports and constants

In [ ]:
import numpy  as np
import pandas as pd
import pyreadstat
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)

WAVES      = [6, 7, 8]
SURVEY_MISSING = [-1, -8, -9]   # ELSA codes: inapplicable / don't know / refusal
ID_COL     = "idauniq"

print("Imports OK | pandas", pd.__version__, "| numpy", np.__version__)

## 2 — Feature lists

We define exactly which columns to pull from each file type.  
This is the single source of truth — update here if the feature set changes.

In [ ]:
# ── CES-D individual items from core wave files ───────────────────────────────
CESD_ITEMS = ["PScedA", "PScedB", "PScedC", "PScedD",
               "PScedE", "PScedF", "PScedG", "PScedH"]

# ── Variables to pull from IFS derived files ──────────────────────────────────
IFS_COLS = [
    # ── Sampling / weights (keep for longitudinal weighting if needed later)
    "lwgt",           # longitudinal weight Wave 1 baseline
    "l4wgt",          # longitudinal weight Wave 4 baseline

    # ── Demographics
    "age",            # age at interview
    "sex",            # sex
    "nonwhite",       # ethnicity binary
    "marstat",        # marital status (merged/computed)
    "couple",         # relationship status
    "died_p",         # partner died since last interview

    # ── Self-rated health
    "srh_hrs",        # self-reported health HRS version (1=excellent, 5=poor)
    "llsill",         # limiting long-standing illness
    "hlimwrk",        # health limits kind/amount of work

    # ── Mobility (will be summed to mobility_count)
    "hemobwa", "hemobsi", "hemobch", "hemobcs", "hemobcl",
    "hemobst", "hemobre", "hemobpu", "hemobli", "hemobpi",
    "hemob96",        # flag: no mobility difficulties

    # ── ADL (Activities of Daily Living)
    "headldr", "headlwa", "headlba", "headlea", "headlbe", "headlwc",

    # ── IADL (Instrumental ADL)
    "headlma", "headlda", "headlpr", "headlsh",
    "headlph", "headlco", "headlme", "headlho", "headlmo",
    "headl96",        # flag: no ADL/IADL difficulties

    # ── Cognitive function
    "memtotb",        # memory index (0-24, excl 2nd prospective test)
    "execnn",         # executive function index (0-20)
    "numtype4",       # numeracy index 4-way split

    # ── Employment
    "ecpos",          # economic activity position
    "worktime",       # full/part time

    # ── Education
    "qual3",          # 3-way qualification split

    # ── Subjective financial / deprivation
    "findiff",        # getting along financially (subjective)
    "ndepriv",        # deprivation index
    "lackresb",       # banded chances of inadequate resources

    # ── Social / household
    "famtype",        # household composition type
    "tenure",         # housing tenure
    "nsibs",          # number of living siblings
    "ngrandch",       # number of grandchildren

    # ── Lifestyle
    "smokerstat",     # smoking status (never/ex/current)

    # ── CES-D summary (IFS pre-computed)
    "cesd_sc",        # CES-D sum score (0-8)
    "cesd_na",        # number of CES-D items answered
]

# ── Variables to pull from financial derived files ────────────────────────────
FIN_COLS = [
    "yq5_bu_s",       # equivalised income quintile (1-5)
    "totwq5_bu_s",    # total non-pension wealth quintile (1-5)
]

print(f"IFS cols: {len(IFS_COLS)} | Financial cols: {len(FIN_COLS)} | CES-D items: {len(CESD_ITEMS)}")

## 3 — Helper functions

In [ ]:
def load_dta(path: Path, columns: list[str], label: str = "") -> pd.DataFrame:
    """
    Load a Stata .dta file via pyreadstat, keeping only requested columns.
    Always includes idauniq. Returns numeric values (no Stata labels/categories).
    """
    cols_to_load = list({ID_COL} | set(columns))

    # First pass: get all column names in the file
    _, meta = pyreadstat.read_dta(str(path), metadataonly=True)
    available = set(meta.column_names)

    # Warn about requested columns that don't exist in this file
    missing_cols = [c for c in cols_to_load if c not in available]
    if missing_cols:
        print(f"  [WARN] {label}: {len(missing_cols)} cols not found: {missing_cols}")
    cols_to_load = [c for c in cols_to_load if c in available]

    df, _ = pyreadstat.read_dta(
        str(path),
        usecols=cols_to_load,
        apply_value_formats=False,   # keep numeric codes, not string labels
        formats_as_category=False,
    )
    return df


def recode_survey_missing(df: pd.DataFrame, missing_codes: list = SURVEY_MISSING) -> pd.DataFrame:
    """
    Replace ELSA survey missing codes (-1, -8, -9) with NaN across the dataframe.
    Only applied to numeric columns.
    """
    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    # Exclude idauniq — it should never be recoded
    numeric_cols = [c for c in numeric_cols if c != ID_COL]
    df[numeric_cols] = df[numeric_cols].replace(missing_codes, np.nan)
    return df


def load_wave(wave: int) -> pd.DataFrame:
    """
    Load and merge all three source files for a single wave.
    Returns a merged dataframe with wave suffix on all feature columns.
    """
    print(f"\n── Wave {wave} ─────────────────────────────")

    # 1. Core wave file — CES-D items only (more features added later)
    print(f"  Loading core...")
    core = load_dta(config.CORE[wave], CESD_ITEMS, label=f"W{wave} core")
    core = recode_survey_missing(core)
    print(f"  Core: {core.shape}")

    # 2. IFS derived file
    print(f"  Loading IFS derived...")
    ifs = load_dta(config.IFS[wave], IFS_COLS, label=f"W{wave} IFS")
    ifs = recode_survey_missing(ifs)
    print(f"  IFS: {ifs.shape}")

    # 3. Financial derived file — quintile vars only
    print(f"  Loading financial derived...")
    fin = load_dta(config.FIN[wave], FIN_COLS, label=f"W{wave} financial")
    fin = recode_survey_missing(fin)
    print(f"  Financial: {fin.shape}")

    # 4. Merge on idauniq — left join keeps all IFS participants
    merged = ifs.merge(core, on=ID_COL, how="left", suffixes=("", "_core"))
    merged = merged.merge(fin,  on=ID_COL, how="left", suffixes=("", "_fin"))
    print(f"  Merged: {merged.shape}")

    # 5. Add wave suffix to all feature columns (not idauniq)
    rename_map = {c: f"{c}_w{wave}" for c in merged.columns if c != ID_COL}
    merged = merged.rename(columns=rename_map)

    return merged


print("Helper functions defined")

## 4 — Load all three waves

In [ ]:
wave_dfs = {}
for w in WAVES:
    wave_dfs[w] = load_wave(w)

print("\nAll waves loaded.")
for w, df in wave_dfs.items():
    print(f"  Wave {w}: {df.shape[0]:,} participants, {df.shape[1]} columns")

## 5 — Build the longitudinal panel

We join all three waves on `idauniq` to get one row per participant.  
Inner join = only participants present in **all three** waves (complete longitudinal sample).  
We also record how many participants drop out at each merge step.

In [ ]:
# Start with wave 6 as the base
panel = wave_dfs[6]
print(f"Wave 6 base: {len(panel):,} participants")

# Merge wave 7
panel = panel.merge(wave_dfs[7], on=ID_COL, how="inner")
print(f"After inner join with Wave 7: {len(panel):,} participants")

# Merge wave 8
panel = panel.merge(wave_dfs[8], on=ID_COL, how="inner")
print(f"After inner join with Wave 8: {len(panel):,} participants")

print(f"\nFinal panel shape: {panel.shape}")
print(f"Attrition from W6 to W6+W7+W8: "
      f"{(1 - len(panel) / len(wave_dfs[6])) * 100:.1f}% of W6 participants dropped")

## 6 — Construct the CES-D target variable

Binary label: depressed at wave 8 = `cesd_sc_w8 >= 3`  
We also check for participants with incomplete CES-D at wave 8 (`cesd_na_w8 < 8`).

In [ ]:
# Participants with a complete CES-D at wave 8
complete_cesd_w8 = panel["cesd_na_w8"] == 8
print(f"Participants with complete CES-D at wave 8: {complete_cesd_w8.sum():,} "
      f"({complete_cesd_w8.mean()*100:.1f}%)")

# Binary depression label
panel["depressed_w8"] = (panel["cesd_sc_w8"] >= 3).astype(int)

# Set label to NaN for those with incomplete CES-D (can't reliably classify)
panel.loc[~complete_cesd_w8, "depressed_w8"] = np.nan

# Class distribution
label_counts = panel["depressed_w8"].value_counts(dropna=False)
print("\nTarget label distribution at Wave 8:")
print(label_counts.rename({0: "Not depressed", 1: "Depressed", np.nan: "Missing CES-D"}))

valid = panel["depressed_w8"].notna()
print(f"\nClass balance (valid labels only): "
      f"{panel.loc[valid, 'depressed_w8'].mean()*100:.1f}% depressed")

## 7 — Missingness overview

In [ ]:
def missingness_report(df: pd.DataFrame, max_rows: int = 40) -> pd.DataFrame:
    """Return a sorted DataFrame showing missing % for each column."""
    miss = (df.isnull().sum() / len(df) * 100).round(1)
    miss = miss[miss > 0].sort_values(ascending=False)
    report = pd.DataFrame({"missing_%": miss, "missing_n": df.isnull().sum()[miss.index]})
    return report.head(max_rows)

miss_report = missingness_report(panel)
print(f"Columns with any missing values: {len(miss_report)}")
print("\nTop columns by missingness:")
miss_report

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot top 30 columns by missingness
top30 = miss_report.head(30)
fig, ax = plt.subplots(figsize=(10, 7))
sns.barplot(x=top30["missing_%"], y=top30.index, ax=ax, color="steelblue")
ax.set_xlabel("Missing (%)")
ax.set_title("Top 30 columns by missingness — longitudinal panel")
ax.axvline(20, color="red", linestyle="--", linewidth=0.8, label="20% threshold")
ax.legend()
plt.tight_layout()
plt.savefig(config.OUTPUTS / "missingness_panel.png", dpi=150)
plt.show()
print("Saved: outputs/missingness_panel.png")

## 8 — Basic sanity checks

In [ ]:
# Check for duplicate participant IDs
dupes = panel[ID_COL].duplicated().sum()
print(f"Duplicate idauniq: {dupes}")
assert dupes == 0, "Duplicate IDs found — investigate before proceeding"

# Age range sanity check
print(f"\nAge at Wave 6: {panel['age_w6'].min():.0f} – {panel['age_w6'].max():.0f} "
      f"(mean {panel['age_w6'].mean():.1f})")

# CES-D score range
for w in WAVES:
    col = f"cesd_sc_w{w}"
    print(f"CES-D score W{w}: {panel[col].min():.0f}–{panel[col].max():.0f} "
          f"(mean {panel[col].mean():.2f}, missing {panel[col].isna().sum()})")

# Sex distribution
print(f"\nSex distribution (W6): {panel['sex_w6'].value_counts().to_dict()}")

print("\nSanity checks passed.")

## 9 — Export

In [ ]:
# Export to parquet — fast, preserves dtypes, good for downstream notebooks
out_path = config.OUTPUTS / "panel_raw.parquet"
panel.to_parquet(out_path, index=False)
print(f"Saved panel to: {out_path}")
print(f"Shape: {panel.shape}")

# Write a loading report
report_path = config.OUTPUTS / "loading_report.txt"
with open(report_path, "w") as f:
    f.write("ELSA Data Loading Report\n")
    f.write("=" * 40 + "\n\n")
    for w in WAVES:
        f.write(f"Wave {w}: {wave_dfs[w].shape[0]:,} participants, {wave_dfs[w].shape[1]} cols\n")
    f.write(f"\nLongitudinal panel (inner join W6+W7+W8): {len(panel):,} participants\n")
    f.write(f"Columns: {panel.shape[1]}\n\n")
    valid_labels = panel["depressed_w8"].notna()
    f.write(f"Valid target labels: {valid_labels.sum():,}\n")
    f.write(f"Depressed at W8 (cesd>=3): {panel.loc[valid_labels, 'depressed_w8'].sum():,} "
            f"({panel.loc[valid_labels, 'depressed_w8'].mean()*100:.1f}%)\n\n")
    f.write("Top 20 columns by missingness:\n")
    f.write(miss_report.head(20).to_string())

print(f"Loading report saved to: {report_path}")

## 10 — Summary

What we've done here:
- Loaded core, IFS, and financial derived files for waves 6, 7, 8
- Recoded ELSA survey missing codes (-1, -8, -9) to NaN throughout
- Merged all sources on `idauniq` within each wave
- Inner-joined across waves to get the complete longitudinal sample
- Constructed the binary target label: `depressed_w8` = (cesd_sc_w8 ≥ 3)
- Produced a missingness overview and sanity checks
- Exported `panel_raw.parquet` for the preprocessing notebook

**Next:** `03_preprocessing.ipynb` — imputation, encoding, scaling, train/test split.